# Plant Disease Classification with EfficientNet-B0

### A Reproducible Deep Learning Baseline for Healthy, Powdery Mildew and Rust Detection

---

This notebook is part of an **agricultural AI prototype** focused on helping farmers identify crop health problems. The platform components include:

- 🔬 **Disease diagnosis** via computer vision
- 📊 **Severity estimation** from leaf images
- 🧠 **Grad-CAM explainability** to visualize model reasoning
- 🌦️ **Weather-based outbreak forecasting** at the district level
- 🗺️ **Privacy-preserving risk maps**
- 💬 **Multilingual agricultural advisory** (English, Urdu, Sindhi)
- 👨‍🔬 **Human expert escalation** for uncertain predictions

**This notebook focuses ONLY on the computer vision disease-classification component.**

We train an **EfficientNet-B0** model with ImageNet transfer learning to classify plant leaf images into three categories: **Healthy**, **Powdery** (mildew), and **Rust**.

> ⚠️ **This notebook is a research/demo baseline. It is not a production agricultural diagnostic system.**


# 1. Objective

### Why Plant Disease Detection Matters

Plant diseases cause billions of dollars in crop losses annually worldwide. In countries like Pakistan, where agriculture employs nearly 40% of the labor force, timely disease identification can prevent devastating yield losses. However, many smallholder farmers lack access to trained agronomists who can diagnose diseases in the field.

### Why Computer Vision Can Help

Convolutional neural networks (CNNs) can learn visual patterns from labelled images of diseased and healthy leaves, enabling automated preliminary screening. A farmer could photograph a leaf and receive an immediate classification result, even without internet connectivity if the model is deployed on-device.

### Why EfficientNet-B0?

- **Compact**: ~5.3M parameters — small enough for mobile deployment.
- **Efficient**: Uses compound scaling to balance depth, width, and resolution.
- **Strong baseline**: Achieves competitive ImageNet accuracy relative to its size.
- **Transfer learning**: ImageNet pretrained features transfer well to plant imagery because low-level texture and color features are domain-general.

### What This Notebook Predicts

The model performs **3-class image classification**:

| Class Index | Label | Description |
|:-:|:--|:--|
| 0 | **Healthy** | No visible disease symptoms |
| 1 | **Powdery** | Powdery mildew infection |
| 2 | **Rust** | Rust fungal infection |

### What This Model Does NOT Do

This classification model does **not** independently determine:
- Treatment dosage or chemical recommendations
- Disease outbreak probability or geographic spread
- Severity validated by certified agronomists
- Multi-disease co-infection detection

Those are separate platform components that sit downstream of this classifier.


## ⚠️ Important Scientific Disclaimer

This dataset is suitable for **demonstrating a machine-learning pipeline** but does not represent the full diversity of Pakistani agricultural field conditions.

Plant images captured in controlled or curated datasets can differ substantially from real farmer photographs because of:

- ☀️ **Lighting** — lab vs. outdoor, time of day, shadows
- 📱 **Camera quality** — smartphone vs. DSLR, resolution, compression
- 🌿 **Background clutter** — soil, other plants, hands, tools
- 🍃 **Leaf orientation** — top vs. underside, curled vs. flat
- 🦠 **Disease stage** — early vs. advanced infection
- 🔀 **Multiple simultaneous diseases** — co-infections common in the field
- 🧬 **Cultivar differences** — leaf shape and color vary by variety
- 🌡️ **Environmental conditions** — drought stress, nutrient deficiency mimicking disease

**Therefore:**

1. Test accuracy in this notebook must **NOT** be presented as Pakistani field accuracy.
2. High benchmark accuracy does **NOT** prove production readiness.
3. Real deployment requires field-collected and agronomist-verified Pakistani data.


# 2. Dataset

The dataset follows a **pre-split** directory structure with separate `Train/`, `Validation/`, and `Test/` folders.

On Kaggle, the dataset is available as **Plant disease recognition dataset** and has a doubled directory layout:

```
/kaggle/input/plant-disease-recognition-dataset/
├── Train/
│   └── Train/
│       ├── Healthy/       (458 images)
│       ├── Powdery/       (430 images)
│       └── Rust/          (434 images)
├── Validation/
│   └── Validation/
│       ├── Healthy/       (20 images)
│       ├── Powdery/       (20 images)
│       └── Rust/          (20 images)
└── Test/
    └── Test/
        ├── Healthy/       (50 images)
        ├── Powdery/       (50 images)
        └── Rust/          (50 images)
```

The notebook automatically resolves this doubled directory structure.

| Split | Healthy | Powdery | Rust | Total |
|:--|:-:|:-:|:-:|:-:|
| Train | 458 | 430 | 434 | **1,322** |
| Validation | 20 | 20 | 20 | **60** |
| Test | 50 | 50 | 50 | **150** |
| **Grand Total** | 528 | 500 | 504 | **1,532** |


## 2.1 Environment Setup & GPU Detection


In [ ]:
import os, sys, time, json, random, logging, warnings
from pathlib import Path
from collections import Counter

import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import matplotlib.gridspec as gridspec

import torch
import torch.nn as nn
import torch.nn.functional as F
from torch.utils.data import Dataset, DataLoader
from torch.optim import AdamW
from torch.optim.lr_scheduler import CosineAnnealingLR

import torchvision
from torchvision import transforms
from torchvision.models import efficientnet_b0, EfficientNet_B0_Weights

from PIL import Image, UnidentifiedImageError
from sklearn.metrics import (
    classification_report, confusion_matrix,
    precision_recall_fscore_support, ConfusionMatrixDisplay,
)

warnings.filterwarnings('ignore', category=UserWarning)
logging.basicConfig(level=logging.INFO, format='%(asctime)s [%(levelname)s] %(message)s')
logger = logging.getLogger('PlantDiseaseClassifier')

# ── Device detection ────────────────────────────────────────────────────────
device = torch.device('cuda' if torch.cuda.is_available() else 'cpu')
use_amp = device.type == 'cuda'

print('=' * 65)
print('Plant Disease Classifier — EfficientNet-B0')
print('=' * 65)
print(f'PyTorch        : {torch.__version__}')
print(f'torchvision    : {torchvision.__version__}')
print(f'Device         : {device}')
if device.type == 'cuda':
    print(f'GPU            : {torch.cuda.get_device_name(0)}')
    print(f'CUDA version   : {torch.version.cuda}')
else:
    print('GPU            : None (CPU fallback — training will be slow)')
print(f'Mixed Precision: {"enabled" if use_amp else "disabled"}')
print('=' * 65)


## 2.2 Automatic Dataset Discovery

The cell below detects the Kaggle dataset path automatically.

Many Kaggle datasets use **doubled directory** structures (e.g. `Train/Train/Healthy/` instead of `Train/Healthy/`). The `resolve_split_dir()` helper transparently handles both layouts.


In [ ]:
KAGGLE_INPUT = Path('/kaggle/input')
DATASET_ROOT = None

# ── Strategy 1: Check known dataset slug ──────────────────────────────
known_path = KAGGLE_INPUT / 'plant-disease-recognition-dataset'
if known_path.exists():
    DATASET_ROOT = known_path

# ── Strategy 2: Recursive search for any dir with Train/Val/Test ─────
if DATASET_ROOT is None and KAGGLE_INPUT.exists():
    for candidate in KAGGLE_INPUT.rglob('*'):
        if (candidate.is_dir()
            and (candidate / 'Train').is_dir()
            and (candidate / 'Validation').is_dir()
            and (candidate / 'Test').is_dir()):
            DATASET_ROOT = candidate
            break

# ── Strategy 3: Local development fallback ───────────────────────────
if DATASET_ROOT is None:
    for fallback in [Path('../data/raw/plant_disease'), Path('data/raw/plant_disease')]:
        if fallback.exists() and (fallback / 'Train').is_dir():
            DATASET_ROOT = fallback
            break

if DATASET_ROOT is None:
    raise FileNotFoundError(
        'ERROR: Could not find dataset. Attach the plant-disease-recognition-dataset to this notebook.'
    )

def resolve_split_dir(root, split_name):
    """Resolve the actual directory containing class folders for a given split.
    
    Handles both layouts:
      - Standard:  root/Train/Healthy/
      - Doubled:   root/Train/Train/Healthy/
    """
    direct = root / split_name
    # Check if class folders exist directly under the split
    if (direct / 'Healthy').is_dir():
        return direct
    # Check for doubled directory (e.g. Train/Train/Healthy/)
    doubled = direct / split_name
    if doubled.is_dir() and (doubled / 'Healthy').is_dir():
        return doubled
    # Fallback: search one level deeper for any subdir containing Healthy/
    for child in direct.iterdir():
        if child.is_dir() and (child / 'Healthy').is_dir():
            return child
    raise FileNotFoundError(f'Cannot find class directories under {direct}')

SPLIT_DIRS = {
    'Train': resolve_split_dir(DATASET_ROOT, 'Train'),
    'Validation': resolve_split_dir(DATASET_ROOT, 'Validation'),
    'Test': resolve_split_dir(DATASET_ROOT, 'Test'),
}

print(f'Dataset root : {DATASET_ROOT.resolve()}')
for sname, sdir in SPLIT_DIRS.items():
    print(f'  {sname:12s}: {sdir}')


## 2.3 Dataset Audit

Before training, we perform a thorough audit of the dataset to verify integrity: image counts per split, class distributions, image dimensions, and corrupt file detection.


In [ ]:
CLASS_NAMES = ['Healthy', 'Powdery', 'Rust']
SPLIT_NAMES = ['Train', 'Validation', 'Test']
VALID_EXTS = {'.jpg', '.jpeg', '.png', '.webp'}

audit_rows = []
all_widths, all_heights = [], []
corrupt_count = 0

for split in SPLIT_NAMES:
    for cls in CLASS_NAMES:
        cls_dir = SPLIT_DIRS[split] / cls
        valid, bad = 0, 0
        if cls_dir.exists():
            for img_file in cls_dir.iterdir():
                if img_file.suffix.lower() in VALID_EXTS:
                    try:
                        with Image.open(img_file) as im:
                            im.verify()
                        with Image.open(img_file) as im:
                            w, h = im.size
                            all_widths.append(w)
                            all_heights.append(h)
                        valid += 1
                    except Exception:
                        bad += 1
        corrupt_count += bad
        audit_rows.append({'Split': split, 'Class': cls, 'Images': valid, 'Corrupt': bad})

df_audit = pd.DataFrame(audit_rows)

# Summary table
pivot = df_audit.pivot_table(index='Split', columns='Class', values='Images', aggfunc='sum')
pivot = pivot[CLASS_NAMES]
pivot['Total'] = pivot.sum(axis=1)
pivot.loc['Grand Total'] = pivot.sum()
print('\n--- Dataset Image Counts ---')
print(pivot.to_string())

# Dimension statistics
print(f'\n--- Image Dimensions ---')
print(f'Min dimensions : {min(all_widths)}x{min(all_heights)}')
print(f'Max dimensions : {max(all_widths)}x{max(all_heights)}')
print(f'Mean dimensions: {np.mean(all_widths):.0f}x{np.mean(all_heights):.0f}')
print(f'Corrupt images : {corrupt_count}')


### Class Distribution Visualization


In [ ]:
# Bar chart of class distribution per split
fig, axes = plt.subplots(1, 3, figsize=(14, 4))
colors = ['#2ecc71', '#f39c12', '#e74c3c']

for idx, split in enumerate(SPLIT_NAMES):
    subset = df_audit[df_audit['Split'] == split]
    axes[idx].bar(subset['Class'], subset['Images'], color=colors)
    axes[idx].set_title(f'{split} Split', fontsize=13, fontweight='bold')
    axes[idx].set_ylabel('Number of Images')
    for i, v in enumerate(subset['Images']):
        axes[idx].text(i, v + 2, str(v), ha='center', fontweight='bold')

plt.suptitle('Dataset Class Distribution', fontsize=15, fontweight='bold')
plt.tight_layout()
plt.show()


### Sample Images

A visual inspection of representative images from each class.


In [ ]:
fig, axes = plt.subplots(2, 3, figsize=(12, 8))

for col_idx, cls in enumerate(CLASS_NAMES):
    cls_dir = SPLIT_DIRS['Train'] / cls
    image_files = sorted([f for f in cls_dir.iterdir() if f.suffix.lower() in VALID_EXTS])
    for row_idx in range(2):
        if row_idx < len(image_files):
            img = Image.open(image_files[row_idx]).convert('RGB')
            axes[row_idx, col_idx].imshow(img)
        axes[row_idx, col_idx].set_title(cls, fontsize=12, fontweight='bold')
        axes[row_idx, col_idx].axis('off')

plt.suptitle('Sample Training Images', fontsize=14, fontweight='bold')
plt.tight_layout()
plt.show()


# 3. Reproducibility

We fix all random seeds to ensure that results can be reproduced across runs. Note that full determinism on CUDA requires `torch.use_deterministic_algorithms(True)`, which can cause performance degradation and may not be compatible with all operations. We seed the generators but allow non-deterministic CUDA kernels for practical training speed.


In [ ]:
SEED = 42

random.seed(SEED)
np.random.seed(SEED)
torch.manual_seed(SEED)
if torch.cuda.is_available():
    torch.cuda.manual_seed_all(SEED)
    # Allow non-deterministic algorithms for performance
    torch.backends.cudnn.deterministic = False
    torch.backends.cudnn.benchmark = True

print(f'Random seed set to: {SEED}')
print(f'PyTorch version  : {torch.__version__}')
print(f'torchvision      : {torchvision.__version__}')
print(f'CUDA available   : {torch.cuda.is_available()}')
if torch.cuda.is_available():
    print(f'GPU name         : {torch.cuda.get_device_name(0)}')
print(f'Device           : {device}')


# 4. Image Preprocessing

EfficientNet-B0 expects **224×224** input images normalized with ImageNet statistics:

- `mean = [0.485, 0.456, 0.406]`
- `std  = [0.229, 0.224, 0.225]`

### Training Augmentation

We apply **conservative data augmentation** to the training set to improve generalization:
- Resize to 256×256, then random crop to 224×224
- Random horizontal flip
- Random vertical flip (p=0.2)
- Random rotation (±15°)
- Color jitter (brightness, contrast, saturation)

### Validation/Test Transforms

Validation and test images use **deterministic** preprocessing only:
- Resize to 256×256
- Center crop to 224×224
- ImageNet normalization

**Augmentation is applied ONLY to training data** because augmented validation/test images would produce misleading performance estimates.


In [ ]:
IMAGE_SIZE = 224
IMAGENET_MEAN = [0.485, 0.456, 0.406]
IMAGENET_STD = [0.229, 0.224, 0.225]

train_transforms = transforms.Compose([
    transforms.Resize((IMAGE_SIZE + 32, IMAGE_SIZE + 32)),
    transforms.RandomCrop(IMAGE_SIZE),
    transforms.RandomHorizontalFlip(),
    transforms.RandomVerticalFlip(p=0.2),
    transforms.RandomRotation(15),
    transforms.ColorJitter(brightness=0.3, contrast=0.3, saturation=0.2, hue=0.05),
    transforms.ToTensor(),
    transforms.Normalize(mean=IMAGENET_MEAN, std=IMAGENET_STD),
])

eval_transforms = transforms.Compose([
    transforms.Resize((IMAGE_SIZE + 32, IMAGE_SIZE + 32)),
    transforms.CenterCrop(IMAGE_SIZE),
    transforms.ToTensor(),
    transforms.Normalize(mean=IMAGENET_MEAN, std=IMAGENET_STD),
])

print('Training transforms:', train_transforms)
print('\nEvaluation transforms:', eval_transforms)


## 4.1 PyTorch Dataset & DataLoaders

We create a custom `Dataset` class that:
- Scans a split directory for images in the three class subdirectories
- Verifies each image can be opened (skips corrupt files)
- Enforces a **deterministic class mapping**: Healthy=0, Powdery=1, Rust=2

The predefined Train/Validation/Test splits are **strictly preserved** — no re-splitting occurs.


In [ ]:
CLASS_TO_IDX = {'Healthy': 0, 'Powdery': 1, 'Rust': 2}

class PlantDiseaseDataset(Dataset):
    """PyTorch Dataset for the pre-split plant disease classification task."""
    
    def __init__(self, split_dir, transform=None):
        self.split_dir = Path(split_dir)
        self.transform = transform
        self.samples = []  # List of (path, label) tuples
        
        for class_name, class_idx in CLASS_TO_IDX.items():
            class_path = self.split_dir / class_name
            if not class_path.exists():
                logger.warning(f'Missing class directory: {class_path}')
                continue
            for img_file in sorted(class_path.iterdir()):
                if img_file.suffix.lower() in VALID_EXTS:
                    try:
                        with Image.open(img_file) as im:
                            im.verify()
                        self.samples.append((img_file, class_idx))
                    except Exception:
                        pass  # Skip corrupt images silently
    
    def __len__(self):
        return len(self.samples)
    
    def __getitem__(self, idx):
        img_path, label = self.samples[idx]
        with Image.open(img_path) as img:
            img = img.convert('RGB')
            if self.transform:
                img = self.transform(img)
        return img, label

# ── Build datasets using resolved split directories ─────────────────────────
train_dataset = PlantDiseaseDataset(SPLIT_DIRS['Train'], transform=train_transforms)
val_dataset   = PlantDiseaseDataset(SPLIT_DIRS['Validation'], transform=eval_transforms)
test_dataset  = PlantDiseaseDataset(SPLIT_DIRS['Test'], transform=eval_transforms)

# ── Configuration ──────────────────────────────────────────────────────────
BATCH_SIZE = 32
NUM_WORKERS = 2
PIN_MEMORY = device.type == 'cuda'

train_loader = DataLoader(train_dataset, batch_size=BATCH_SIZE, shuffle=True,
                          num_workers=NUM_WORKERS, pin_memory=PIN_MEMORY)
val_loader   = DataLoader(val_dataset, batch_size=BATCH_SIZE, shuffle=False,
                          num_workers=NUM_WORKERS, pin_memory=PIN_MEMORY)
test_loader  = DataLoader(test_dataset, batch_size=BATCH_SIZE, shuffle=False,
                          num_workers=NUM_WORKERS, pin_memory=PIN_MEMORY)

print(f'Train samples     : {len(train_dataset)}')
print(f'Validation samples: {len(val_dataset)}')
print(f'Test samples      : {len(test_dataset)}')
print(f'Batch size        : {BATCH_SIZE}')
print(f'Class mapping     : {CLASS_TO_IDX}')


# 5. EfficientNet-B0 Transfer Learning

### Architecture Overview

EfficientNet-B0 is the baseline model from the [EfficientNet](https://arxiv.org/abs/1905.11946) family (Tan & Le, 2019). It uses **compound scaling** — simultaneously scaling network depth, width, and input resolution — to achieve better accuracy per FLOP than prior architectures.

| Property | Value |
|:--|:--|
| Parameters | ~5.3M |
| Input size | 224×224 |
| ImageNet Top-1 | ~77.1% |
| Backbone output | 1280-dim feature vector |

### Custom Classification Head

We replace the original 1000-class ImageNet head with:

```
Dropout(p=0.3, inplace=True)
Linear(1280, 3)
```

This head is compatible with the backend inference engine.


In [ ]:
def build_model(num_classes=3, pretrained=True, dropout=0.3):
    """Build EfficientNet-B0 with custom classification head.
    
    Architecture matches backend/ml/disease_classifier/model.py exactly.
    """
    weights = EfficientNet_B0_Weights.DEFAULT if pretrained else None
    model = efficientnet_b0(weights=weights)
    in_features = model.classifier[1].in_features  # 1280
    model.classifier = nn.Sequential(
        nn.Dropout(p=dropout, inplace=True),
        nn.Linear(in_features, num_classes),
    )
    return model

NUM_CLASSES = 3
DROPOUT = 0.3

model = build_model(num_classes=NUM_CLASSES, pretrained=True, dropout=DROPOUT).to(device)

# Print model summary
total_params = sum(p.numel() for p in model.parameters())
trainable_params = sum(p.numel() for p in model.parameters() if p.requires_grad)
print(f'Total parameters     : {total_params:,}')
print(f'Trainable parameters : {trainable_params:,}')
print(f'Classifier head      : {model.classifier}')


## 5.1 Training Configuration

All hyperparameters are defined here for easy modification.


In [ ]:
# ╔══════════════════════════════════════════════════════════════════╗
# ║  TRAINING HYPERPARAMETERS — Modify these as needed              ║
# ╚══════════════════════════════════════════════════════════════════╝

EPOCHS        = 20
LEARNING_RATE = 1e-4
WEIGHT_DECAY  = 0.01
LABEL_SMOOTH  = 0.1
PATIENCE      = 7     # Early stopping patience (epochs without improvement)

# ── Optimizer & Scheduler ──────────────────────────────────────────────────
criterion = nn.CrossEntropyLoss(label_smoothing=LABEL_SMOOTH)
optimizer = AdamW(model.parameters(), lr=LEARNING_RATE, weight_decay=WEIGHT_DECAY)
scheduler = CosineAnnealingLR(optimizer, T_max=EPOCHS, eta_min=1e-6)
scaler = torch.amp.GradScaler('cuda', enabled=use_amp)

print(f'Epochs         : {EPOCHS}')
print(f'Learning rate  : {LEARNING_RATE}')
print(f'Weight decay   : {WEIGHT_DECAY}')
print(f'Label smoothing: {LABEL_SMOOTH}')
print(f'Early stopping : patience={PATIENCE}')
print(f'Optimizer      : AdamW')
print(f'Scheduler      : CosineAnnealingLR (T_max={EPOCHS})')
print(f'Loss           : CrossEntropyLoss')
print(f'Mixed precision: {use_amp}')


# 6. Model Training

The training loop:
1. Trains on the **Train** split with augmentation.
2. Evaluates on the **Validation** split after every epoch.
3. Saves the **best checkpoint** based on highest validation accuracy.
4. Implements **early stopping** to prevent overfitting.
5. Uses **Automatic Mixed Precision (AMP)** on CUDA for faster training.

The **Test** set is not touched until final evaluation.


In [ ]:
OUTPUT_DIR = Path('/kaggle/working')
OUTPUT_DIR.mkdir(parents=True, exist_ok=True)
PT_PATH   = OUTPUT_DIR / 'classifier.pt'
JSON_PATH = OUTPUT_DIR / 'classifier_classes.json'

best_val_acc = 0.0
best_val_loss = float('inf')
best_epoch = 0
epochs_no_improve = 0
history = []

print('=' * 75)
print(f'Training: {len(train_dataset)} images | Validation: {len(val_dataset)} images')
print(f'Classes : {NUM_CLASSES} — {CLASS_NAMES}')
print(f'Device  : {device}')
print(f'Epochs  : {EPOCHS} | Batch size: {BATCH_SIZE}')
print('=' * 75)

for epoch in range(1, EPOCHS + 1):
    t0 = time.time()
    
    # ── Train phase ─────────────────────────────────────────────────────
    model.train()
    train_loss, train_correct, train_total = 0.0, 0, 0
    for images, labels in train_loader:
        images, labels = images.to(device), labels.to(device)
        optimizer.zero_grad()
        with torch.amp.autocast('cuda', enabled=use_amp):
            outputs = model(images)
            loss = criterion(outputs, labels)
        scaler.scale(loss).backward()
        scaler.step(optimizer)
        scaler.update()
        train_loss += loss.item() * images.size(0)
        train_correct += (outputs.argmax(1) == labels).sum().item()
        train_total += images.size(0)
    
    avg_train_loss = train_loss / train_total
    avg_train_acc  = train_correct / train_total
    
    # ── Validation phase ────────────────────────────────────────────────
    model.eval()
    val_loss, val_correct, val_total = 0.0, 0, 0
    with torch.no_grad():
        for images, labels in val_loader:
            images, labels = images.to(device), labels.to(device)
            with torch.amp.autocast('cuda', enabled=use_amp):
                outputs = model(images)
                loss = criterion(outputs, labels)
            val_loss += loss.item() * images.size(0)
            val_correct += (outputs.argmax(1) == labels).sum().item()
            val_total += images.size(0)
    
    avg_val_loss = val_loss / val_total
    avg_val_acc  = val_correct / val_total
    current_lr = scheduler.get_last_lr()[0]
    scheduler.step()
    elapsed = time.time() - t0
    
    # ── Logging ─────────────────────────────────────────────────────────
    improved = ''
    if avg_val_acc > best_val_acc or (avg_val_acc == best_val_acc and avg_val_loss < best_val_loss):
        best_val_acc = avg_val_acc
        best_val_loss = avg_val_loss
        best_epoch = epoch
        epochs_no_improve = 0
        improved = ' ★ BEST'
        
        # Save checkpoint in backend-compatible format
        checkpoint_payload = {
            'state_dict': model.state_dict(),
            'model_state_dict': model.state_dict(),
            'class_names': CLASS_NAMES,
            'num_classes': NUM_CLASSES,
            'image_size': IMAGE_SIZE,
            'normalization': {'mean': IMAGENET_MEAN, 'std': IMAGENET_STD},
            'model_architecture': 'efficientnet_b0',
            'training_config': {
                'epochs': EPOCHS, 'batch_size': BATCH_SIZE,
                'learning_rate': LEARNING_RATE, 'weight_decay': WEIGHT_DECAY,
                'dropout': DROPOUT, 'label_smoothing': LABEL_SMOOTH,
                'image_size': IMAGE_SIZE,
            },
            'epoch': epoch,
            'val_acc': avg_val_acc,
            'val_loss': avg_val_loss,
            'best_val_accuracy': avg_val_acc,
            'seed': SEED,
            'training_seed': SEED,
        }
        torch.save(checkpoint_payload, PT_PATH)
    else:
        epochs_no_improve += 1
    
    history.append({
        'epoch': epoch,
        'train_loss': avg_train_loss, 'train_acc': avg_train_acc,
        'val_loss': avg_val_loss, 'val_acc': avg_val_acc,
        'lr': current_lr,
    })
    
    print(f'Epoch {epoch:02d}/{EPOCHS} | '
          f'Train Loss: {avg_train_loss:.4f} Acc: {avg_train_acc:.4f} | '
          f'Val Loss: {avg_val_loss:.4f} Acc: {avg_val_acc:.4f} | '
          f'LR: {current_lr:.6f} | {elapsed:.1f}s{improved}')
    
    if epochs_no_improve >= PATIENCE:
        print(f'\nEarly stopping triggered after {PATIENCE} epochs without improvement.')
        break

# ── Save class names JSON ───────────────────────────────────────────────
with open(JSON_PATH, 'w', encoding='utf-8') as f:
    json.dump(CLASS_NAMES, f, indent=2)

# ── Save training history ───────────────────────────────────────────────
df_history = pd.DataFrame(history)
df_history.to_csv(OUTPUT_DIR / 'training_log.csv', index=False)

print(f'\nTraining complete. Best epoch: {best_epoch} (Val Acc: {best_val_acc:.4f})')
print(f'Checkpoint saved : {PT_PATH}')
print(f'Class names saved: {JSON_PATH}')


# 7. Training Curves

Visualizing loss and accuracy curves helps identify:
- **Overfitting**: Training metrics improve while validation plateaus or worsens.
- **Underfitting**: Both training and validation metrics remain poor.
- **Good convergence**: Both curves improve and stabilize together.


In [ ]:
fig, axes = plt.subplots(1, 3, figsize=(18, 5))

# Loss curves
axes[0].plot(df_history['epoch'], df_history['train_loss'], 'o-', label='Train Loss', color='#3498db')
axes[0].plot(df_history['epoch'], df_history['val_loss'], 's-', label='Val Loss', color='#e74c3c')
axes[0].axvline(x=best_epoch, color='gray', linestyle='--', alpha=0.5, label=f'Best (epoch {best_epoch})')
axes[0].set_title('Loss Curves', fontsize=13, fontweight='bold')
axes[0].set_xlabel('Epoch')
axes[0].set_ylabel('Cross-Entropy Loss')
axes[0].legend()
axes[0].grid(True, alpha=0.3)

# Accuracy curves
axes[1].plot(df_history['epoch'], df_history['train_acc'], 'o-', label='Train Acc', color='#3498db')
axes[1].plot(df_history['epoch'], df_history['val_acc'], 's-', label='Val Acc', color='#e74c3c')
axes[1].axvline(x=best_epoch, color='gray', linestyle='--', alpha=0.5, label=f'Best (epoch {best_epoch})')
axes[1].set_title('Accuracy Curves', fontsize=13, fontweight='bold')
axes[1].set_xlabel('Epoch')
axes[1].set_ylabel('Accuracy')
axes[1].legend()
axes[1].grid(True, alpha=0.3)

# Learning rate
axes[2].plot(df_history['epoch'], df_history['lr'], 'o-', color='#2ecc71')
axes[2].set_title('Learning Rate Schedule', fontsize=13, fontweight='bold')
axes[2].set_xlabel('Epoch')
axes[2].set_ylabel('Learning Rate')
axes[2].grid(True, alpha=0.3)

plt.suptitle('Training Progress', fontsize=15, fontweight='bold')
plt.tight_layout()
plt.savefig(OUTPUT_DIR / 'training_curves.png', dpi=150, bbox_inches='tight')
plt.show()


# 8. Loading the Best Checkpoint

We reload the saved best checkpoint from disk to ensure we evaluate the exact weights that achieved the highest validation accuracy — not the final epoch weights.


In [ ]:
# Reconstruct model architecture and load best weights
best_model = build_model(num_classes=NUM_CLASSES, pretrained=False, dropout=DROPOUT).to(device)
ckpt = torch.load(PT_PATH, map_location=device, weights_only=False)
best_model.load_state_dict(ckpt['state_dict'])
best_model.eval()

# Verify checkpoint integrity
required_keys = {'state_dict', 'class_names', 'num_classes'}
assert required_keys.issubset(ckpt.keys()), f'Checkpoint missing keys: {required_keys - ckpt.keys()}'
assert ckpt['class_names'] == CLASS_NAMES, f'Class name mismatch: {ckpt["class_names"]}'
assert ckpt['num_classes'] == NUM_CLASSES, f'Class count mismatch: {ckpt["num_classes"]}'

print(f'Loaded best checkpoint from epoch {ckpt["epoch"]}')
print(f'Validation accuracy: {ckpt["val_acc"]:.4f}')
print(f'Class names: {ckpt["class_names"]}')
print(f'Checkpoint verified: compatible with backend.')


# 8. Final Test Evaluation

The test set is used **only once** for final performance estimation. It was never seen during training or model selection.


In [ ]:
all_preds = []
all_labels = []
all_probs = []

with torch.no_grad():
    for images, labels in test_loader:
        images = images.to(device)
        with torch.amp.autocast('cuda', enabled=use_amp):
            outputs = best_model(images)
        probs = F.softmax(outputs, dim=1)
        preds = probs.argmax(dim=1)
        all_preds.extend(preds.cpu().numpy())
        all_labels.extend(labels.numpy())
        all_probs.extend(probs.cpu().numpy())

all_preds = np.array(all_preds)
all_labels = np.array(all_labels)
all_probs = np.array(all_probs)

test_accuracy = (all_preds == all_labels).mean()
macro_p, macro_r, macro_f1, _ = precision_recall_fscore_support(all_labels, all_preds, average='macro')
weighted_p, weighted_r, weighted_f1, _ = precision_recall_fscore_support(all_labels, all_preds, average='weighted')
report_str = classification_report(all_labels, all_preds, target_names=CLASS_NAMES, digits=4)
cm = confusion_matrix(all_labels, all_preds)

print('=' * 65)
print('PLANT-DISEASE DATASET TEST-SET PERFORMANCE')
print('=' * 65)
print(f'Top-1 Accuracy    : {test_accuracy:.4f} ({test_accuracy*100:.2f}%)')
print(f'Macro Precision   : {macro_p:.4f}')
print(f'Macro Recall      : {macro_r:.4f}')
print(f'Macro F1-Score    : {macro_f1:.4f}')
print(f'Weighted F1-Score : {weighted_f1:.4f}')
print(f'\nClassification Report:\n{report_str}')

# Save reports
with open(OUTPUT_DIR / 'classification_report.txt', 'w', encoding='utf-8') as f:
    f.write('Plant-Disease dataset test-set performance\n')
    f.write(f'Top-1 Accuracy: {test_accuracy:.4f}\n')
    f.write(f'Macro F1: {macro_f1:.4f}\n')
    f.write(f'Weighted F1: {weighted_f1:.4f}\n\n')
    f.write(report_str)

df_cm = pd.DataFrame(cm, index=CLASS_NAMES, columns=CLASS_NAMES)
df_cm.to_csv(OUTPUT_DIR / 'confusion_matrix_test.csv')


### Confusion Matrix


In [ ]:
fig, ax = plt.subplots(figsize=(7, 6))
disp = ConfusionMatrixDisplay(confusion_matrix=cm, display_labels=CLASS_NAMES)
disp.plot(ax=ax, cmap='Blues', values_format='d')
ax.set_title('Test Set Confusion Matrix', fontsize=14, fontweight='bold')
plt.tight_layout()
plt.savefig(OUTPUT_DIR / 'confusion_matrix.png', dpi=150, bbox_inches='tight')
plt.show()


## 8.1 Per-Class Performance Analysis


In [ ]:
per_class = precision_recall_fscore_support(all_labels, all_preds, average=None, labels=[0,1,2])
df_per_class = pd.DataFrame({
    'Class': CLASS_NAMES,
    'Precision': per_class[0],
    'Recall': per_class[1],
    'F1-Score': per_class[2],
    'Support': per_class[3],
})
df_per_class = df_per_class.round(4)
print(df_per_class.to_string(index=False))


# 9. Visual Predictions

Showing both **correct** and **incorrect** predictions with their confidence scores helps assess the model's practical behaviour.


In [ ]:
def denormalize(tensor):
    """Undo ImageNet normalization for visualization."""
    mean = torch.tensor(IMAGENET_MEAN).view(3, 1, 1)
    std = torch.tensor(IMAGENET_STD).view(3, 1, 1)
    return (tensor.cpu() * std + mean).clamp(0, 1)

# Find correct and incorrect predictions
correct_idx = np.where(all_preds == all_labels)[0]
incorrect_idx = np.where(all_preds != all_labels)[0]

def show_predictions(indices, title, max_show=6):
    n = min(len(indices), max_show)
    if n == 0:
        print(f'No {title.lower()} to display.')
        return
    fig, axes = plt.subplots(1, n, figsize=(3.5 * n, 4))
    if n == 1:
        axes = [axes]
    for i, idx in enumerate(indices[:n]):
        img_tensor, _ = test_dataset[idx]
        img_vis = denormalize(img_tensor).permute(1, 2, 0).numpy()
        axes[i].imshow(img_vis)
        pred_cls = CLASS_NAMES[all_preds[idx]]
        true_cls = CLASS_NAMES[all_labels[idx]]
        conf = all_probs[idx].max()
        color = 'green' if all_preds[idx] == all_labels[idx] else 'red'
        axes[i].set_title(f'Pred: {pred_cls} ({conf:.2f})\nTrue: {true_cls}',
                         fontsize=10, color=color, fontweight='bold')
        axes[i].axis('off')
    plt.suptitle(title, fontsize=14, fontweight='bold')
    plt.tight_layout()
    plt.show()

show_predictions(correct_idx, 'Correct Predictions (Green)', max_show=6)
show_predictions(incorrect_idx, 'Incorrect Predictions (Red)', max_show=6)


# 10. Prediction Confidence Analysis

Model confidence (softmax probability) should be interpreted with caution:
- High confidence ≠ guaranteed correctness
- Neural network softmax outputs are often **miscalibrated** — they tend to be overconfident
- A 95% confidence score does not mean 95% real-world accuracy

The threshold below (0.50) is an **engineering threshold** for flagging uncertain predictions, not a scientifically validated uncertainty estimate.


In [ ]:
CONFIDENCE_THRESHOLD = 0.50

max_confidences = all_probs.max(axis=1)

fig, axes = plt.subplots(1, 2, figsize=(14, 5))

# Confidence distribution
axes[0].hist(max_confidences, bins=30, color='#3498db', edgecolor='white', alpha=0.8)
axes[0].axvline(x=CONFIDENCE_THRESHOLD, color='red', linestyle='--', label=f'Threshold ({CONFIDENCE_THRESHOLD})')
axes[0].set_title('Prediction Confidence Distribution', fontsize=13, fontweight='bold')
axes[0].set_xlabel('Max Softmax Probability')
axes[0].set_ylabel('Count')
axes[0].legend()

# Confidence by correctness
correct_conf = max_confidences[all_preds == all_labels]
incorrect_conf = max_confidences[all_preds != all_labels]
axes[1].hist(correct_conf, bins=20, alpha=0.7, label='Correct', color='#2ecc71')
if len(incorrect_conf) > 0:
    axes[1].hist(incorrect_conf, bins=20, alpha=0.7, label='Incorrect', color='#e74c3c')
axes[1].set_title('Confidence: Correct vs Incorrect', fontsize=13, fontweight='bold')
axes[1].set_xlabel('Max Softmax Probability')
axes[1].set_ylabel('Count')
axes[1].legend()

plt.tight_layout()
plt.show()

low_conf = (max_confidences < CONFIDENCE_THRESHOLD).sum()
print(f'Predictions below confidence threshold ({CONFIDENCE_THRESHOLD}): {low_conf}/{len(max_confidences)}')
print(f'Mean confidence: {max_confidences.mean():.4f}')
print(f'Min confidence : {max_confidences.min():.4f}')
print(f'Max confidence : {max_confidences.max():.4f}')


# 11. Explainability with Grad-CAM

**Grad-CAM** (Gradient-weighted Class Activation Mapping) highlights which regions of an image most influenced the model's prediction. It computes the gradient of the predicted class score with respect to the feature maps of the last convolutional layer.

### Important Caveats
- Grad-CAM is an **interpretability aid**, not proof that the model understands disease biology.
- It shows correlation, not causation.
- Highlighted regions may correspond to texture, color, or shape cues rather than pathological features.


In [ ]:
# Grad-CAM implementation for EfficientNet-B0
class GradCAM:
    """Minimal Grad-CAM implementation targeting the last conv block."""
    
    def __init__(self, model, target_layer):
        self.model = model
        self.target_layer = target_layer
        self.gradients = None
        self.activations = None
        self._register_hooks()
    
    def _register_hooks(self):
        def forward_hook(module, input, output):
            self.activations = output.detach()
        def backward_hook(module, grad_in, grad_out):
            self.gradients = grad_out[0].detach()
        self.target_layer.register_forward_hook(forward_hook)
        self.target_layer.register_full_backward_hook(backward_hook)
    
    def generate(self, input_tensor, class_idx=None):
        self.model.eval()
        output = self.model(input_tensor)
        if class_idx is None:
            class_idx = output.argmax(dim=1).item()
        
        self.model.zero_grad()
        target = output[0, class_idx]
        target.backward()
        
        # Weight activations by mean gradient
        weights = self.gradients.mean(dim=[2, 3], keepdim=True)
        cam = (weights * self.activations).sum(dim=1, keepdim=True)
        cam = F.relu(cam)
        cam = F.interpolate(cam, size=input_tensor.shape[2:], mode='bilinear', align_corners=False)
        cam = cam.squeeze().cpu().numpy()
        cam = (cam - cam.min()) / (cam.max() - cam.min() + 1e-8)
        return cam

# Target the last convolutional block of EfficientNet-B0
target_layer = best_model.features[-1]
grad_cam = GradCAM(best_model, target_layer)

# Generate Grad-CAM for sample test images
fig, axes = plt.subplots(3, 3, figsize=(12, 12))
sample_indices = []
for cls_idx in range(3):
    cls_mask = np.where(all_labels == cls_idx)[0]
    if len(cls_mask) > 0:
        sample_indices.append(cls_mask[0])

for row, idx in enumerate(sample_indices[:3]):
    img_tensor, label = test_dataset[idx]
    input_t = img_tensor.unsqueeze(0).to(device)
    
    # Original image
    img_vis = denormalize(img_tensor).permute(1, 2, 0).numpy()
    axes[row, 0].imshow(img_vis)
    axes[row, 0].set_title(f'Original ({CLASS_NAMES[label]})', fontweight='bold')
    axes[row, 0].axis('off')
    
    # Grad-CAM heatmap
    cam = grad_cam.generate(input_t)
    axes[row, 1].imshow(cam, cmap='jet')
    axes[row, 1].set_title('Grad-CAM Heatmap', fontweight='bold')
    axes[row, 1].axis('off')
    
    # Overlay
    axes[row, 2].imshow(img_vis)
    axes[row, 2].imshow(cam, cmap='jet', alpha=0.4)
    pred = all_preds[idx]
    conf = all_probs[idx].max()
    axes[row, 2].set_title(f'Overlay — Pred: {CLASS_NAMES[pred]} ({conf:.2f})', fontweight='bold')
    axes[row, 2].axis('off')

plt.suptitle('Grad-CAM Explainability', fontsize=15, fontweight='bold')
plt.tight_layout()
plt.savefig(OUTPUT_DIR / 'gradcam_examples.png', dpi=150, bbox_inches='tight')
plt.show()


# 12. Exporting the Model

The checkpoint is saved in a format compatible with the FastAPI backend. The backend's `load_checkpoint()` function expects:
- `state_dict`: Model weights
- `class_names`: List of class name strings
- `num_classes`: Integer class count


In [ ]:
output_files = {
    'classifier.pt': 'Model checkpoint (weights + metadata)',
    'classifier_classes.json': 'Class names list',
    'training_log.csv': 'Epoch-by-epoch training history',
    'classification_report.txt': 'Sklearn test evaluation report',
    'confusion_matrix_test.csv': 'Test confusion matrix',
    'training_curves.png': 'Loss/accuracy plots',
    'confusion_matrix.png': 'Confusion matrix visualization',
    'gradcam_examples.png': 'Grad-CAM explainability examples',
}

print('--- Output Files ---')
for fname, purpose in output_files.items():
    fpath = OUTPUT_DIR / fname
    if fpath.exists():
        size_kb = fpath.stat().st_size / 1024
        print(f'  [OK] {fname:35s} {size_kb:8.1f} KB  — {purpose}')
    else:
        print(f'  [--] {fname:35s}       ---  — {purpose}')

# Verify checkpoint can be loaded
verify_ckpt = torch.load(PT_PATH, map_location='cpu', weights_only=False)
assert 'state_dict' in verify_ckpt
assert 'class_names' in verify_ckpt
assert 'num_classes' in verify_ckpt
print('\nCheckpoint integrity verification: PASSED')


# 13. System Integration

### How the Trained Model Enters the Platform

After training on Kaggle, download the output files and place them in the backend repository:

```
Kaggle:                          Local Repository:
/kaggle/working/classifier.pt  → backend/ml/models/classifier.pt
/kaggle/working/classifier_classes.json → backend/ml/models/classifier_classes.json
```

Then set the environment variable:
```
DIAGNOSIS_MOCK_MODE=false
```

### Full Inference Pipeline

```
Farmer uploads leaf image
        ↓
FastAPI  POST /api/v1/analyze
        ↓
InferenceEngine loads classifier.pt
        ↓
EfficientNet-B0 forward pass
        ↓
Disease name + Confidence score
        ↓
Severity Estimator (green-pixel analysis)
        ↓
Grad-CAM overlay generation
        ↓
Structured JSON API response
        ↓
Next.js mobile-first Results UI
```


# 14. Limitations

This experiment has important limitations that must be acknowledged:

| # | Limitation | Impact |
|:-:|:--|:--|
| 1 | **Small dataset** (1,532 images) | May not generalize to diverse field conditions |
| 2 | **Low dataset diversity** | Controlled/curated images differ from real photos |
| 3 | **Controlled conditions** | Lab lighting ≠ field lighting |
| 4 | **Domain shift** | Model trained on one distribution, deployed on another |
| 5 | **No Pakistani field images** | Performance on Pakistani crops is unknown |
| 6 | **Only 3 classes** | Real agriculture has hundreds of diseases |
| 7 | **No multi-disease detection** | Co-infections are common in the field |
| 8 | **No temporal progression** | Disease stages are not modelled |
| 9 | **No validated severity** | Severity estimation is not independently verified |
| 10 | **No treatment recommendation** | This model classifies, it does not prescribe |
| 11 | **No outbreak prediction** | Weather/geographic risk is a separate component |
| 12 | **Confidence ≠ certainty** | Softmax probabilities are miscalibrated |

> **This model is a baseline component, not a complete agricultural decision-support system.**


# 15. Future Work

- 🇵🇰 **Pakistani field image collection**: Partner with local agricultural extension services to gather real farmer photographs across cotton, wheat, rice, and sugarcane.
- 👨‍🔬 **Expert-verified labels**: Ensure every training image is validated by certified agronomists.
- 🌾 **More crops & diseases**: Expand beyond 3 classes to cover Pakistan's major crops.
- 🎨 **Field-condition augmentation**: Simulate phone camera noise, variable lighting, motion blur.
- 🔄 **Domain adaptation**: Fine-tune on Pakistani field data to bridge the domain gap.
- 📐 **Calibration**: Apply temperature scaling or Platt scaling to produce reliable confidence estimates.
- ❓ **Uncertainty estimation**: Monte Carlo dropout or ensemble methods for flagging uncertain predictions.
- 🔬 **Better severity segmentation**: Use pixel-level segmentation rather than green-pixel heuristics.
- 🛰️ **Satellite/weather integration**: Combine satellite imagery and weather data for outbreak forecasting.
- 🗺️ **District-level risk mapping**: Aggregate anonymized diagnoses into geographic risk dashboards.
- 🌐 **Multilingual advisory**: RAG-based agricultural guidance in English, Urdu, and Sindhi.
- 🧑‍⚕️ **Human expert escalation**: Route low-confidence or critical diagnoses to agricultural officers.


# 16. Responsible AI Considerations

### Safety
- The system must **never** make unsafe pesticide dosage recommendations based solely on AI predictions.
- Low-confidence predictions must be **routed to human experts** rather than presented as definitive diagnoses.

### Transparency
- Model outputs should be presented as **assistance** rather than unquestionable truth.
- Training data limitations, benchmark conditions, and model uncertainty must be **disclosed** to users.
- Synthetic or benchmark-derived results must **never** be presented as field validation.

### Privacy
- Farmer images and location data require **privacy protection**.
- Exact GPS coordinates must **never** be exposed publicly on risk maps.
- Location data should be fuzzed to district-level granularity.

### Equity
- The system should work across different smartphone cameras and network conditions.
- Multilingual support ensures farmers who do not speak English can still benefit.


# 17. Reproducing This Experiment

### Step-by-Step Instructions

1. **Upload the dataset** to Kaggle as a new dataset with the directory structure:
   ```
   plant_disease/
   ├── Train/{Healthy,Powdery,Rust}/
   ├── Validation/{Healthy,Powdery,Rust}/
   └── Test/{Healthy,Powdery,Rust}/
   ```

2. **Create a new Kaggle Notebook** and attach the dataset.

3. **Set Accelerator to GPU** (Settings → Accelerator → GPU T4 x2 or P100).

4. **Upload this notebook** or copy-paste the cells.

5. **Run All Cells** and wait for training to complete (~5–15 min on GPU).

6. **Review** training curves, test metrics, and Grad-CAM visualizations.

7. **Download** `classifier.pt` and `classifier_classes.json` from the Output section.

8. **Deploy** by placing files into the backend repository:
   ```
   backend/ml/models/classifier.pt
   backend/ml/models/classifier_classes.json
   ```

9. Set `DIAGNOSIS_MOCK_MODE=false` and start the FastAPI server.


# 18. Results Summary

The following summary is **generated from the actual experiment**. No values are hardcoded.


In [ ]:
gpu_name = torch.cuda.get_device_name(0) if torch.cuda.is_available() else 'CPU'

print('=' * 65)
print('EXPERIMENT RESULTS SUMMARY')
print('=' * 65)
print(f'Model              : EfficientNet-B0 (ImageNet pretrained)')
print(f'Classes            : {NUM_CLASSES} — {CLASS_NAMES}')
print(f'Training images    : {len(train_dataset)}')
print(f'Validation images  : {len(val_dataset)}')
print(f'Test images        : {len(test_dataset)}')
print(f'GPU                : {gpu_name}')
print(f'Best epoch         : {best_epoch}')
print(f'Best val accuracy  : {best_val_acc:.4f}')
print(f'Test accuracy      : {test_accuracy:.4f} ({test_accuracy*100:.2f}%)')
print(f'Macro F1-Score     : {macro_f1:.4f}')
print(f'Weighted F1-Score  : {weighted_f1:.4f}')
print('=' * 65)
print()
print('These results represent performance on this dataset only and')
print('should not be interpreted as Pakistani field performance.')


# 19. Output Files


In [ ]:
print('=' * 75)
print('DOWNLOAD THESE FILES FROM THE KAGGLE OUTPUT SECTION')
print('=' * 75)

download_files = [
    ('classifier.pt', 'Model checkpoint — deploy to backend/ml/models/'),
    ('classifier_classes.json', 'Class names — deploy to backend/ml/models/'),
    ('training_log.csv', 'Epoch-by-epoch training/validation metrics'),
    ('classification_report.txt', 'Test set sklearn classification report'),
    ('confusion_matrix_test.csv', 'Test set confusion matrix (CSV)'),
    ('training_curves.png', 'Training/validation loss and accuracy plots'),
    ('confusion_matrix.png', 'Confusion matrix heatmap'),
    ('gradcam_examples.png', 'Grad-CAM explainability visualizations'),
]

for fname, purpose in download_files:
    fpath = OUTPUT_DIR / fname
    if fpath.exists():
        size_kb = fpath.stat().st_size / 1024
        status = 'EXISTS'
    else:
        size_kb = 0
        status = 'MISSING'
    print(f'  [{status:7s}] {fname:35s} {size_kb:8.1f} KB')
    print(f'            Purpose: {purpose}')


## Conclusion

This notebook demonstrates a **reproducible EfficientNet-B0 transfer-learning baseline** for three plant-health classes (Healthy, Powdery, Rust).

The model is **one component** of a broader agricultural intelligence architecture, whose long-term objective is to combine:

| Component | Purpose |
|:--|:--|
| 🔬 Computer Vision | Leaf disease classification |
| 🧠 Explainability | Grad-CAM visualization of model reasoning |
| 📊 Severity Estimation | Quantifying affected leaf area |
| 🌦️ Weather-Based Risk | District-level disease outbreak forecasting |
| 🗺️ Geospatial Mapping | Privacy-preserving risk maps |
| 💬 Multilingual Advisory | RAG-based guidance in English, Urdu, Sindhi |
| 👨‍🔬 Human Expert Escalation | Routing uncertain predictions to agronomists |

**Safety, transparency, and farmer privacy remain central to the system.**

---

*Empowering farmers with responsible agricultural intelligence.*
